In [1]:
import os
import numpy as np
import pandas as pd
import healpy as hp
from astropy.time import Time, TimeDelta
from astropy.coordinates import SkyCoord
from astroplan import Observer
import astropy.units as u

from rubin_nights import connections
from rubin_nights import lfa_data
from rubin_nights import plot_utils as rn_plots

from IPython.display import display, HTML
import matplotlib.pyplot as plt

import pickle
from lsst.resources import ResourcePath
from rubin_scheduler.scheduler.schedulers import CoreScheduler
from rubin_scheduler.scheduler.features import Conditions
from rubin_scheduler.scheduler.model_observatory import ModelObservatory
from rubin_scheduler.utils import ddf_locations

import rubin_scheduler.scheduler.basis_functions as basis_functions
import rubin_scheduler.scheduler.detailers as detailers
from rubin_scheduler.skybrightness_pre import dark_m5
from rubin_scheduler.site_models import SeeingModel

import schedview.compute as schedview_compute


In [2]:
endpoints_dev = connections.get_clients(tokenfile=os.path.join(os.path.expanduser("~"), ".lsst/usdf_rsp"), site='usdf-dev')
endpoints_dev

endpoints = endpoints_dev

In [3]:
#day_obs = Time(Time.now().mjd - 0.5, format='mjd', scale='tai').iso[0:10]
day_obs = "2026-06-29"

queue = 1


day_obs_time = Time(f"{day_obs}T12:00:00", format='isot', scale='tai')
tnow = Time.now()
observer = Observer.at_site('Rubin')
sunset = Time(observer.sun_set_time(day_obs_time, which='next', horizon=-0*u.deg), format='jd')
sunrise = Time(observer.sun_rise_time(day_obs_time, which='next', horizon=-0*u.deg), format='jd')
print(day_obs, 'sunset', sunset.iso,  'sunrise', sunrise.iso, 'now', Time.now().iso)

topic = "lsst.sal.Scheduler.logevent_target"
#topic = "lsst.sal.Scheduler.logevent_largeFileObjectAvailable"
targets = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
for c in ['ra', 'decl', 'skyAngle']:
    targets[c] = targets[c].astype(float)
if len(targets) == 0:
    display(endpoints['efd'].select_top_n(topic, '*', 2, index=queue))
print(len(targets))

topic = "lsst.sal.Scheduler.logevent_observation"
observations = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
for c in ['ra', 'decl', 'rotSkyPos']:
    observations[c] = observations[c].astype(float)
print(len(observations))

topic = "lsst.sal.Scheduler.logevent_largeFileObjectAvailable"
snapshots = endpoints['efd'].select_time_series(topic, '*', sunset, sunrise, index=queue)
print(len(snapshots))

if len(targets) > 0 and len(observations) > 0:
    to = pd.merge_asof(
                targets.sort_values("targetId").reset_index("time"),
                observations.sort_values("targetId").reset_index("time"),
                on="targetId",
                left_by=["ra", "decl", "skyAngle"],
                right_by=["ra", "decl", "rotSkyPos"],
                suffixes=("", "_o"),
                allow_exact_matches=True,
                direction="forward",
            )
    to.sort_values(by="time", inplace=True)
    to = to.astype({"targetId": int, "blockId": int, "skyAngle": float})
    print(len(to))
elif len(targets) > 0:
    to = pd.DataFrame(targets.sort_values("targetId").reset_index("time"))
    to['time_o'] = np.nan
    print("targets only")
else:
    to = None

2026-06-29 sunset 2026-06-29 21:48:43.224 sunrise 2026-06-30 11:44:28.099 now 2026-06-30 20:29:35.993
886
875
891
886


In [4]:
visits = endpoints['consdb'].get_visits("lsstcam", sunset, sunrise)

In [5]:
programs = ["BLOCK-365", "BLOCK-407", "BLOCK-408", "BLOCK-416"]
visits.query("science_program in @programs")[['target_name', 'visit_id', 'obs_start',  'observation_reason', 'science_program', 'band',  'zero_point_1s_pred', 'fwhm_eff', 'clouds']]


,target_name,visit_id,obs_start,observation_reason,science_program,band,zero_point_1s_pred,fwhm_eff,clouds
10,lowdust,2026062900045,2026-06-29T22:51:19.330000,twilight_near_sun,BLOCK-407,r,28.376665,2.027672,0.060814
11,"nes, lowdust",2026062900046,2026-06-29T22:51:47.622000,twilight_near_sun,BLOCK-407,r,28.360304,2.228412,0.082849
12,nes,2026062900047,2026-06-29T22:52:16.287000,twilight_near_sun,BLOCK-407,r,28.342320,2.060431,0.048547
13,lowdust,2026062900048,2026-06-29T22:52:52.253000,twilight_near_sun,BLOCK-407,r,28.350811,1.692259,0.008915
14,lowdust,2026062900049,2026-06-29T22:53:20.510000,twilight_near_sun,BLOCK-407,r,28.340926,1.790782,0.021362
...,...,...,...,...,...,...,...,...,...
880,nes,2026062900915,2026-06-30T10:42:10.439000,twilight_near_sun,BLOCK-407,r,28.361989,1.360721,-0.022051
881,nes,2026062900916,2026-06-30T10:42:39.872000,twilight_near_sun,BLOCK-407,r,28.358934,1.339038,-0.021399
882,nes,2026062900917,2026-06-30T10:43:07.478000,twilight_near_sun,BLOCK-407,r,28.343403,1.360780,-0.013822
883,nes,2026062900918,2026-06-30T10:43:38.636000,twilight_near_sun,BLOCK-407,r,28.354687,1.421419,0.016755


In [6]:
ack = to.groupby("snapshotUri").first()[['time', 'ra', 'decl', 'skyAngle', 'filter', 'note', 'targetName', 'airmass', 'targetId', 'blockId']]

In [11]:
to["snapshotUri"].iloc[668]

'https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T07:30:07.244.p'

In [74]:
ack.shape

(884, 10)

In [77]:
ack.iloc[845:880]


,time,ra,decl,skyAngle,filter,note,targetName,airmass,targetId,blockId
snapshotUri,,,,,,,,,,
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T09:52:49.772.p,2026-06-30 09:56:32.047814+00:00,9.221615,-43.931937,271.813129,i,"DD:ELAISS1, 3900","ddf_elaiss1, lowdust",1.047691,23554,117886
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T09:52:51.808.p,2026-06-30 09:57:14.305151+00:00,336.541332,-26.042034,338.757095,z,greedy z,lowdust,1.044578,23555,117887
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T09:52:54.253.p,2026-06-30 10:00:15.758250+00:00,333.083581,-25.474442,338.979683,z,greedy z,lowdust,1.069966,23556,117888
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:00:54.663.p,2026-06-30 10:00:58.244665+00:00,21.916662,-48.346489,176.495862,i,greedy i,lowdust,1.103689,23557,117889
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:00:56.871.p,2026-06-30 10:05:10.819130+00:00,15.873041,-43.756655,178.078363,i,greedy i,lowdust,1.060459,23558,117890
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:00:58.925.p,2026-06-30 10:05:55.058958+00:00,23.267661,-45.724636,169.804895,i,greedy i,lowdust,1.107860,23559,117891
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:06:34.873.p,2026-06-30 10:10:08.063366+00:00,340.007763,-26.459993,337.504729,z,greedy z,lowdust,1.047583,23560,117892
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:06:37.396.p,2026-06-30 10:10:52.372235+00:00,341.086956,-29.094480,329.129006,z,greedy z,lowdust,1.043825,23561,117893
https://s3.cp.lsst.org/rubinobs-lfa-cp/Scheduler:1/Scheduler:1/2026/06/29/Scheduler:1_Scheduler:1_2026-06-30T10:06:39.726.p,2026-06-30 10:11:39.658224+00:00,338.981077,-23.828555,344.402586,z,greedy z,lowdust,1.057528,23562,117894


In [63]:
to[["time", "targetId"]][200:250]

,time,targetId
868,2026-06-30 01:16:09.185934+00:00,27628
869,2026-06-30 01:17:00.452873+00:00,27629
870,2026-06-30 01:17:44.711662+00:00,27630
871,2026-06-30 01:18:32.607842+00:00,27631
872,2026-06-30 01:19:21.891423+00:00,27632
873,2026-06-30 01:20:08.180900+00:00,27633
874,2026-06-30 01:20:52.823195+00:00,27634
875,2026-06-30 01:21:40.128496+00:00,27635
876,2026-06-30 01:22:25.436586+00:00,27636
877,2026-06-30 01:23:14.056399+00:00,27637


In [73]:
visits.iloc[849:900][['obs_start', 'visit_id',  'band',  "scheduler_note"]]


,obs_start,visit_id,band,scheduler_note
849,2026-06-30T10:01:02.690000,2026062900884,i,"DD:ELAISS1, 3900"
850,2026-06-30T10:05:15.436000,2026062900885,z,greedy z
851,2026-06-30T10:05:59.253000,2026062900886,z,greedy z
852,2026-06-30T10:10:12.343000,2026062900887,i,greedy i
853,2026-06-30T10:10:56.449000,2026062900888,i,greedy i
854,2026-06-30T10:11:43.583000,2026062900889,i,greedy i
855,2026-06-30T10:15:58.977000,2026062900890,z,greedy z
856,2026-06-30T10:16:43.944000,2026062900891,z,greedy z
857,2026-06-30T10:17:38.290000,2026062900892,z,greedy z
858,2026-06-30T10:21:51.497000,2026062900893,i,greedy i


In [26]:
to.columns

Index(['time', 'airmass', 'alt', 'az', 'blockId', 'cloud', 'decl',
       'exposureTimes0', 'exposureTimes1', 'exposureTimes2', 'exposureTimes3',
       'exposureTimes4', 'exposureTimes5', 'exposureTimes6', 'exposureTimes7',
       'exposureTimes8', 'exposureTimes9', 'filter', 'isSequence', 'moonAlt',
       'moonAz', 'moonDec', 'moonDistance', 'moonPhase', 'moonRa', 'note',
       'numExposures', 'numProposals', 'offsetX', 'offsetY',
       'private_efdStamp', 'private_identity', 'private_kafkaStamp',
       'private_origin', 'private_rcvStamp', 'private_revCode',
       'private_seqNum', 'private_sndStamp', 'proposalId0', 'proposalId1',
       'proposalId2', 'proposalId3', 'proposalId4', 'ra', 'requestMjd',
       'requestTime', 'rotAngle', 'salIndex', 'schedulerNote', 'seeing',
       'sequenceDuration', 'sequenceNVisits', 'sequenceVisits', 'skyAngle',
       'skyBrightness', 'slewTime', 'snapshotUri', 'solarElong', 'sunAlt',
       'sunAz', 'sunDec', 'sunRa', 'targetId', 'targetNam

In [68]:
visits.shape

(917, 250)